# Collect EWS Usage Data from Audit Logs

This notebook will collect app registration data from Entra and signin audit logs from Microsoft 365 for applications with EWS permissions. The data will be stored in CSV files in the `$OutputPath` folder and can be easily imported in Excel or Power BI for further analysis.

Included in this repo is a second notebook `Report-EWS-App-Usage.ipynb` that can be used to review the data as well.

In [ ]:
# Script to call Find-EwsUsage.ps1 

# Configure Log Level
$VerbosePreference = "SilentlyContinue"

# Ensure Script can run in the current context
Set-ExecutionPolicy -ExecutionPolicy Bypass -Scope Process -Force

In [ ]:
# Import Utilities Module
Import-Module ./Modules/EwsUtilities.psm1

In [ ]:
# Load non-secret configuration (for OutputPath only)
$appSettings = Get-Config

if ($appSettings -and -not [string]::IsNullOrWhiteSpace($appSettings.OutputPath)) {
    $OutputPath = $appSettings.OutputPath
    Write-Host "Using OutputPath from configuration file"
} else {
    $OutputPath = ".\Usage-Data"
    Write-Host "Using default OutputPath: $OutputPath"
}

function Get-RequiredEnvironmentVariable {
    param(
        [Parameter(Mandatory = $true)][string]$Name
    )

    $value = [Environment]::GetEnvironmentVariable($Name, "Process")
    if ([string]::IsNullOrWhiteSpace($value)) {
        $value = [Environment]::GetEnvironmentVariable($Name, "User")
    }

    if ([string]::IsNullOrWhiteSpace($value)) {
        throw "Missing required environment variable '$Name'. Run ./Scripts/New-EwsUsageAuditApp.ps1 first."
    }

    return $value
}

$TenantId = Get-RequiredEnvironmentVariable -Name "EWS_USAGE_TENANT_ID"
$AuditAppId = Get-RequiredEnvironmentVariable -Name "EWS_USAGE_AUDIT_APP_ID"
$AuditAppSecret = Get-RequiredEnvironmentVariable -Name "EWS_USAGE_AUDIT_APP_SECRET"

Write-Host "Loaded tenant and app credentials from environment variables."

# Ensure the output directory exists
if (-not (Test-Path -Path $OutputPath)) {
    New-Item -ItemType Directory -Path $OutputPath | Out-Null
}


## Find EWS Usage Data

In this section `Find-EwsUsage.ps1` is used to collect EWS activity data based on Entra app registrations with EWS permissions and audit log sign in data for those applications. The data is stored in CSV files in the `$OutputPath` folder.

In [ ]:

# Call Find-EwsUsage.ps1 with the required parameters
Write-Host "Calling Find-EwsUsage.ps1 with AppId, TenantId, and ClientSecret..."
#& $FindEwsUsageScriptPath `
./Scripts/Find-EwsUsage.ps1 `
    -OutputPath $OutputPath `
    -OAuthClientId $AuditAppId `
    -OAuthTenantId $TenantId `
    -OAuthClientSecret (ConvertTo-SecureString -String $AuditAppSecret -AsPlainText -Force) `
    -Operation GetEwsActivity `

# Check if the script executed successfully
if ($LASTEXITCODE -eq 0) {
    Write-Host "Find-EwsUsage.ps1 executed successfully. Check the output at: $OutputPath"
} else {
    Write-Host "Find-EwsUsage.ps1 execution failed. Please check the parameters and try again."
}